### Global Setup & Hyperparameters
Centralized configuration block defining directory paths, pipeline thresholds, feature engineering constants, and model hyperparameters (LightGBM, XGBoost, CatBoost).

In [1]:
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
from pathlib import Path
from tqdm import tqdm

import polars as pl
import numpy as np
import pandas as pd
import gc
import glob
import csv
import sys

import lightgbm as lgb
import xgboost as xgb

In [2]:
TIME_BUCKET_PREFIXES = (
    "days30_", "days90_", "days120_", "days180_", "days360_",
    "for3years_", "forquarter_", "foryear_", "formonth_", "forweek_", "fortoday_",
    "firstquarter_", "secondquarter_", "thirdquarter_", "fourthquarter_",
)

BINARY_FLAG_SUBSTRINGS = (
    "isbidproduct", "isdebitcard", "isreference", "remitter",
    "safeguarantyflag", "opencred", "mastercontrexist", "mastercontrelectronic",
    "equalityempfrom", "equalitydataagreement", "contaddr_matchlist",
    "contaddr_smempladdr",
)

In [3]:
DATA_DIR = "/kaggle/input/competitions/home-credit-credit-risk-model-stability/"
OUTPUT_DIR = Path("/kaggle/working/")
CURATED_DIR = OUTPUT_DIR / "curated"
LOGS_DIR = OUTPUT_DIR / "logs"
MODELS_DIR = OUTPUT_DIR / "models"
SUBMISSIONS_DIR = OUTPUT_DIR / "submissions"

# Create output directories
for dir_path in [CURATED_DIR, MODELS_DIR, SUBMISSIONS_DIR, LOGS_DIR]:
    dir_path.mkdir(exist_ok=True, parents=True)


# ---------------------------- PIPELINE SETTINGS ----------------------------
N_FOLDS = 5
RANDOM_STATE = 42

# Feature selection thresholds
CORRELATION_THRESHOLD = 0.95
NULL_THRESHOLD = 0.95

# Memory optimization
LOW_MEMORY_MODE = True
MAX_FEATURES_PER_TABLE = 50
CORRELATION_SAMPLE_SIZE = 10000


# ---------------------------- MODEL HYPERPARAMETERS ----------------------------
NUM_BOOST_ROUND = 1500
EARLY_STOPPING_ROUNDS = 100

LGBM_PARAMS = {
    'objective':         'binary',
    'metric':            'auc',
    'boosting_type':     'gbdt',
    'verbosity':         -1,
    'seed':              RANDOM_STATE,
    'is_unbalance':      True,
    'num_leaves':        31,
    'max_depth':         -1,
    'learning_rate':     0.05,
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'lambda_l1':         0.1,
    'lambda_l2':         0.1,
    'min_gain_to_split': 0.02,
    'min_data_in_leaf':  20,
    'device':            'cpu',
}


XGB_PARAMS = {
    'objective':        'binary:logistic',
    'eval_metric':      'auc',
    'eta':              0.05,
    'max_depth':        6,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 20,
    'lambda':           1.0,
    'alpha':            0.1,
    'tree_method':      'hist',     
    'device':           'cuda',
    'verbosity':        0,
    'seed':             RANDOM_STATE,
    'enable_categorical': True,
}


CATBOOST_PARAMS = {
    'iterations':         NUM_BOOST_ROUND + 1000,
    'learning_rate':      0.05,
    'depth':              6,
    'loss_function':      'Logloss',
    'eval_metric':        'AUC',
    'random_seed':        42,
    'od_type':            'Iter',
    'verbose':            100,
    'auto_class_weights': 'Balanced',
    'bootstrap_type':     'Bernoulli',
    'subsample':          0.8,
    'task_type':          'GPU',
    'devices':            '0:1',
}

### Data Loading & Preprocessing Utilities
This section defines utility functions to efficiently read, concatenate, and preprocess Parquet files using Polars. It ensures data type consistency by strictly enforcing the training schema onto the test set and safely converts string features into Pandas categorical types.

In [ ]:
def set_table_dtypes(df: pl.DataFrame, schema=None) -> pl.DataFrame:
    if schema is not None:
        schema_to_apply = {col: dtype for col, dtype in schema.items() if col in df.columns}
        return df.cast(schema_to_apply, strict=False)
    
    for col in df.columns:
        # if col == "isdebitcard_527L":
        #     df = df.with_columns(pl.col(col).cast(pl.Boolean).alias(col))
        if col[-1] in ("P", "A"):
            df = df.with_columns(pl.col(col).cast(pl.Float32).alias(col))

    return df

def convert_strings(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        if df[col].dtype.name in ['object', 'string', 'bool']:
            df[col] = df[col].astype("string").astype('category')
            current_categories = df[col].cat.categories
            new_categories = current_categories.to_list() + ["Unknown"]
            new_dtype = pd.CategoricalDtype(categories=new_categories, ordered=True)
            df[col] = df[col].astype(new_dtype)
            df[col] = df[col].fillna("Unknown")

    return df

def read_files(pattern_path, schema=None):
    files = glob.glob(pattern_path)
    if not files:
        return None
    
    df = pl.concat(
        [pl.read_parquet(f).pipe(set_table_dtypes) for f in files],
        how="vertical_relaxed"
    )
    if schema:
        df = set_table_dtypes(df, schema)

    return df
    

In [5]:
train_base = read_files(DATA_DIR + "parquet_files/train/train_base.parquet")
train_static = read_files(DATA_DIR + "parquet_files/train/train_static_0_*.parquet")
train_static_cb = read_files(DATA_DIR + "parquet_files/train/train_static_cb_0.parquet")
train_person_1 = read_files(DATA_DIR + "parquet_files/train/train_person_1.parquet")
train_applprev_1 = read_files(DATA_DIR + "parquet_files/train/train_applprev_1_0.parquet")
# train_applprev_2 = read_files(DATA_DIR + "parquet_files/train/train_applprev_2.parquet")
train_debitcard = read_files(DATA_DIR + "parquet_files/train/train_debitcard_1.parquet")
train_deposit = read_files(DATA_DIR + "parquet_files/train/train_deposit_1.parquet")
train_tax_registry_a_1 = read_files(DATA_DIR + "parquet_files/train/train_tax_registry_a_1.parquet")
train_tax_registry_b_1 = read_files(DATA_DIR + "parquet_files/train/train_tax_registry_b_1.parquet")
train_credit_bureau_a_1 = read_files(DATA_DIR + "parquet_files/train/train_credit_bureau_a_1_1.parquet")

In [6]:
test_base = read_files(DATA_DIR + "parquet_files/test/test_base.parquet", schema=train_base.schema)
test_static = read_files(DATA_DIR + "parquet_files/test/test_static_0_*.parquet", schema=train_static.schema)
test_static_cb = read_files(DATA_DIR + "parquet_files/test/test_static_cb_0.parquet", schema=train_static_cb.schema)
test_person_1 = read_files(DATA_DIR + "parquet_files/test/test_person_1.parquet", schema=train_person_1.schema)
test_applprev_1 = read_files(DATA_DIR + "parquet_files/test/test_applprev_1_0.parquet", schema=train_applprev_1.schema)
# test_applprev_2 = read_files(DATA_DIR + "parquet_files/test/test_applprev_2.parquet", schema=train_applprev_2.schema)
test_debitcard = read_files(DATA_DIR + "parquet_files/test/test_debitcard_1.parquet", schema=train_debitcard.schema)
test_deposit = read_files(DATA_DIR + "parquet_files/test/test_deposit_1.parquet", schema=train_deposit.schema)
test_tax_registry_a_1 = read_files(DATA_DIR + "parquet_files/test/test_tax_registry_a_1.parquet", schema=train_tax_registry_a_1.schema)
test_tax_registry_b_1 = read_files(DATA_DIR + "parquet_files/test/test_tax_registry_b_1.parquet", schema=train_tax_registry_b_1.schema)
test_credit_bureau_a_1 = read_files(DATA_DIR + "parquet_files/test/test_credit_bureau_a_1_1.parquet", schema=train_credit_bureau_a_1.schema)

In [7]:
table_pairs = [
    ("base", train_base, test_base),
    ("static", train_static, test_static),
    ("static_cb", train_static_cb, test_static_cb),
    ("person_1", train_person_1, test_person_1),
    ("applprev_1", train_applprev_1, test_applprev_1),
    # ("applprev_2", train_applprev_2, test_applprev_2),
    ("debitcard", train_debitcard, test_debitcard),
    ("deposit", train_deposit, test_deposit),
    ("tax_a", train_tax_registry_a_1, test_tax_registry_a_1),
    ("tax_b", train_tax_registry_b_1, test_tax_registry_b_1),
    ("bureau_a_1", train_credit_bureau_a_1, test_credit_bureau_a_1),
]

for name, train_df, test_df in table_pairs:
    if train_df is None or test_df is None:
        print(f"MISSING file {name}")
        continue
    common_cols = [c for c in train_df.columns if c in test_df.columns]
    
    print(f"{name:<22}: Train shape {train_df.shape} x Test shape {test_df.shape}")
    for col in common_cols:
        if train_df[col].dtype != test_df[col].dtype:
            print(f"{col} Train: {train_df[col].dtype}, Test:  {test_df[col].dtype}")

base                  : Train shape (1526659, 5) x Test shape (10, 4)
static                : Train shape (1526659, 168) x Test shape (30, 168)
static_cb             : Train shape (1500476, 53) x Test shape (10, 53)
person_1              : Train shape (2973991, 37) x Test shape (10, 37)
applprev_1            : Train shape (3887684, 41) x Test shape (10, 41)
debitcard             : Train shape (157302, 6) x Test shape (10, 6)
deposit               : Train shape (145086, 5) x Test shape (10, 5)
tax_a                 : Train shape (3275770, 5) x Test shape (10, 5)
tax_b                 : Train shape (1107933, 5) x Test shape (10, 5)
bureau_a_1            : Train shape (6009192, 79) x Test shape (10, 79)


### Feature Engineering
This section uses Polars to automate statistical aggregations, calculate financial ratios, and process date differences.

In [8]:
def process_dates(df: pl.DataFrame) -> pl.DataFrame:
    date_cols = [c for c in df.columns if c[-1] == "D"]

    
    df = df.with_columns([pl.col(c).str.to_date("%Y-%m-%d") for c in (date_cols + ["date_decision"])])
    
    exprs = []
    for col in date_cols:
        if col == "date_decision":
            continue
        exprs.append((pl.col("date_decision") - pl.col(col)).dt.total_days().fill_null(-1).alias(f"days_since_{col}"))

    df = df.with_columns(exprs).drop(date_cols)
    return df

def aggregate_helper(col: str = "case_id", ):
    suffix = col[-1]
    exprs = []

    if suffix == "A":
        exprs.extend([
            pl.col(col).mean().alias(f"{col}_mean"),
            pl.col(col).max().alias(f"{col}_max"),
            pl.col(col).min().alias(f"{col}_min"),
            pl.col(col).std().alias(f"{col}_std"),
            pl.col(col).sum().alias(f"{col}_sum"),
            pl.col(col).tail(3).mean().alias(f"{col}_last3_mean"),
            pl.col(col).count().alias(f"{col}_count"),
            pl.col(col).diff().mean().alias(f"{col}_trend_slope")
        ])
    
    elif suffix == "P":
        exprs.extend([
            pl.col(col).mean().alias(f"{col}_mean"),
            pl.col(col).max().alias(f"{col}_max"),
            pl.col(col).std().alias(f"{col}_std"),
            pl.col(col).tail(3).mean().alias(f"{col}_last3_mean"),
            pl.col(col).count().alias(f"{col}_count"),
            pl.col(col).diff().mean().alias(f"{col}_trend_slope")
        ])

    elif suffix == "L":
        if any(s in col for s in BINARY_FLAG_SUBSTRINGS):
            exprs += [pl.col(col).max().alias(f"{col}_ever")]
        elif any(col.startswith(p) for p in TIME_BUCKET_PREFIXES):
            exprs += [pl.col(col).last().alias(f"{col}_last")]
        else:
            exprs.extend([
                pl.col(col).mean().alias(f"{col}_mean"),
                pl.col(col).max().alias(f"{col}_max"),
                pl.col(col).std().alias(f"{col}_std"),
                pl.col(col).tail(3).mean().alias(f"{col}_last3_mean"),
                pl.col(col).count().alias(f"{col}_count"),
            ])

    elif suffix == "T":
        exprs.extend([
            pl.col(col).drop_nulls().first().alias(f"{col}_mode"),
            pl.col(col).n_unique().alias(f"{col}_nunique"),
        ])

    elif col.startswith("days_since_"):
        exprs.extend([
            pl.col(col).mean().alias(f"{col}_mean"),
            pl.col(col).max().alias(f"{col}_max"),
            pl.col(col).min().alias(f"{col}_min"),
            pl.col(col).tail(3).mean().alias(f"{col}_last3_mean"),
            pl.col(col).count().alias(f"{col}_count"),
            pl.col(col).diff().mean().alias(f"{col}_trend_slope")
        ])
        
    return exprs

def ratio_feature(df: pl.DataFrame):
    exprs = []
    cols = set(df.columns)
    exprs = []

    def find(col: str) -> str | None:
        if col in cols:
            return col
        if f"{col}_sum" in cols:
            return f"{col}_sum"
        return None

    def safe_ratio(num_col: str, den_col: str, alias: str):
        a = find(num_col)
        b = find(den_col)
        if a and b and df[a].dtype.is_numeric() and df[b].dtype.is_numeric():
            exprs.append((pl.col(a) / (pl.col(b) + 1e-6)).alias(alias))


    safe_ratio("credacc_transactions_402L", "credacc_credlmt_575A", "ratio_cash_to_credit_limit")       # Cash drawings activity to credit limit (applprev_1)
    safe_ratio("avgpmtlast12m_4525200A", "currdebt_22A", "ratio_payment_to_balance")                    # Payment to outstanding balance (static)
    safe_ratio("annuity_780A", "maininc_215A", "ratio_dti")                                             # DTI — annuity instalment vs main income (static)
    safe_ratio("totalsettled_863A", "credamount_770A", "ratio_principal_repaid")                        # Principal repaid: total settled vs credit amount (static)
    safe_ratio("currdebt_22A", "credamount_770A", "ratio_debt_to_credit")                               # Current debt utilisation vs credit amount (static)
    safe_ratio("maxdpdlast12m_727P", "numinstls_657L", "ratio_dpd_per_instalment")                      # DPD per instalment — delinquency severity (static)
    safe_ratio("numinstpaidearly_338L", "numinstls_657L", "ratio_early_payment_rate")                   # Early payment rate — financial discipline (static)
    safe_ratio("numinstlsallpaid_934L", "numinstls_657L", "ratio_on_time_payment_rate")                 # On-time payment rate (static)
    safe_ratio("credacc_actualbalance_314A", "credacc_credlmt_575A", "ratio_balance_to_credit_limit")   # Credit card utilisation: balance vs credit limit (applprev_1)
    a = find("credamount_590A")                                                                         # Remaining debt ratio: (loan - debt) / loan (applprev_1)
    b = find("currdebt_94A")
    if a and b:
        exprs.append(
            ((pl.col(a) - pl.col(b)) / (pl.col(a) + 1e-6)).alias("ratio_remaining_debt")
        )
        
    df = df.with_columns(exprs)
    return df

In [9]:
def aggregate(df: pl.DataFrame, group_col: str="case_id") -> pl.DataFrame:
    df = ratio_feature(df)
    feature_cols = [c for c in df.columns if (c[-1] in ["A", "P", "L", "T"] or c.startswith("days_since_") or c.startswith("trend_")) and c != group_col]

    exprs = []
    for col in feature_cols:
        exprs.extend(aggregate_helper(col))

    if not exprs:
        return df.select(group_col).unique
    
    result = df.group_by(group_col).agg(exprs)
    
    return result

In [10]:
def feature_engineering(base_df: pl.DataFrame, table: dict, schema_dict: dict=None):
    result = base_df.select(["case_id", "date_decision"])
    
    if schema_dict is None:
        schema_dict = {}
        is_train = True
    else:
        is_train = False

    for name, df in table.items():
        if df is None:
            continue
        df_agg = aggregate(df)
        
        if LOW_MEMORY_MODE:
            if not is_train:
                schema = schema_dict.get(name)
                if schema:
                    cols_to_select = [c for c in schema.keys() if c in df_agg.columns]
                    df_agg = df_agg.select(cols_to_select)
                    df_agg = df_agg.cast(schema)
            else:
                if len(df_agg.columns) > MAX_FEATURES_PER_TABLE:
                    num_cols = [c for c, dtype in df_agg.schema.items() if dtype.is_numeric() and c != "case_id"]
                    vars = df_agg.select([pl.col(c).var().alias(c) for c in num_cols]).to_dicts()[0]
                    
                    top_cols = sorted(vars.keys(), key=lambda x: vars[x] or -1, reverse=True)[:MAX_FEATURES_PER_TABLE]
                    
                    final_cols = ["case_id"] + [c for c in df_agg.columns if c not in num_cols] + top_cols
                    df_agg = df_agg.select(list(dict.fromkeys(final_cols)))
                
                schema_dict[name] = df_agg.schema

        df_agg = df_agg.rename({c: f"{name}_{c}" for c in df_agg.columns if c != "case_id"})
        result = result.join(df_agg, on="case_id", how="left")
        
        table[name] = None
        del df_agg
        gc.collect()
        
    result = process_dates(result)
    
    return result.drop("date_decision"), schema_dict

### Memory Downcasting
`shrink_dtypes` reduces the dataframe's memory footprint by downcasting numeric columns. It safely converts `Int64` features to `Int32` based on their min/max bounds and casts all `Float64` columns to `Float32`.

In [11]:
def shrink_dtypes(df: pl.DataFrame) -> pl.DataFrame:
    """
    Shrink dtypes to reduce memory usage
    """

    expressions = []
    for col in df.columns:
        dtype = df[col].dtype
        if dtype == pl.Int64:
            if df[col].min() is not None and -2147483648 <= df[col].min() and df[col].max() <= 2147483647:
                expressions.append(pl.col(col).cast(pl.Int32))
        elif dtype == pl.Float64:
            expressions.append(pl.col(col).cast(pl.Float32))
    
    return df.with_columns(expressions)

`filter_features` function removes highly sparse features by dropping columns where the missing value ratio exceeds a specified threshold.

In [12]:
def filter_features(df_train, df_test, null_threshold=0.95):
    total_rows = len(df_train)
    null_ratios = df_train.select([
        (pl.col(c).null_count() / total_rows).alias(c) 
        for c in df_train.columns
    ])

    cols_to_drop = [
        col for col in df_train.columns 
        if null_ratios[col][0] > null_threshold
    ]
    # Ignore meta columns
    cols_to_drop = [c for c in cols_to_drop if c not in meta_cols]
    
    print(f"Null Filter: Dropping {len(cols_to_drop)} columns with >{null_threshold*100:.0f}% nulls")
    
    df_train = df_train.drop(cols_to_drop)
    df_test = df_test.drop([c for c in cols_to_drop if c in df_test.columns])
    
    return df_train, df_test

`filter_correlated_features` mitigates multicollinearity by identifying and dropping numeric features that exceed a specified correlation threshold.

In [13]:
def filter_correlated_features(df_train, df_test, correlation_threshold=0.95):
    numeric_cols = [c for c, dtype in df_train.schema.items() 
                    if dtype.is_numeric() and c not in meta_cols]

    data_sample = df_train.select(numeric_cols).sample(
        n=min(CORRELATION_SAMPLE_SIZE, len(df_train)), 
        seed=42
    ).to_pandas()
    
    non_constant_cols = data_sample.columns[data_sample.var() > 1e-9].tolist()
    corr_matrix = data_sample[non_constant_cols].corr().abs()
    
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    cols_to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    
    print(f"Correlation Filter: Dropping {len(cols_to_drop)} columns")
    
    df_train = df_train.drop(cols_to_drop)
    df_test = df_test.drop([c for c in cols_to_drop if c in df_test.columns])
    
    return df_train, df_test

### Dataset Assembly & Integration
This section orchestrates the final dataset construction by processing date features in static tables and executing the aggregation pipeline on 1-to-N historical records. It seamlessly left-joins all processed features onto the base tables while strictly enforcing the training schema onto the test set to ensure perfect alignment.

In [14]:
meta_cols = ['case_id', 'target', 'WEEK_NUM', 'MONTH', 'date_decision']

In [15]:
if train_static is not None:
    train_static_proc = process_dates(train_static.join(train_base.select(["case_id","date_decision"]), on="case_id", how="left"))
    train_static_proc = train_static_proc.drop("date_decision")
    del train_static
else:
    train_static_proc = None

if train_static_cb is not None:
    train_static_cb_proc = process_dates(train_static_cb.join(train_base.select(["case_id", "date_decision"]),on="case_id", how="left"))
    train_static_cb_proc = train_static_cb_proc.drop("date_decision")
    del train_static_cb
else:
    train_static_cb_proc = None

train_tables_1n = {
    "person1":          train_person_1,
    "applprev_1":       train_applprev_1,
    # "applprev_2":       train_applprev_2,
    "debitcard":        train_debitcard,
    "deposit":          train_deposit,
    "tax_reg_a":        train_tax_registry_a_1,
    "tax_reg_b":        train_tax_registry_b_1,
    "bureau_a_1":       train_credit_bureau_a_1,
}

train_table, saved_schema = feature_engineering(train_base, train_tables_1n)

train_data = train_base
if train_static_proc is not None:
    train_data = train_data.join(train_static_proc, on="case_id", how="left")
if train_static_cb_proc is not None:
    train_data = train_data.join(train_static_cb_proc, on="case_id", how="left")
if train_table is not None:
    train_data = train_data.join(train_table, on="case_id", how="left")

if "date_decision" in train_data.columns:
    train_data = train_data.drop("date_decision")

print(f"Train shape: {train_data.shape}")
gc.collect()

Train shape: (1526659, 487)


0

In [16]:
train_data = shrink_dtypes(train_data)

In [17]:
if test_static is not None:
    test_static_proc = process_dates(test_static.join(test_base.select(["case_id","date_decision"]), on="case_id", how="left"))
    test_static_proc = test_static_proc.drop("date_decision")
    del test_static
else:
    test_static_proc = None

if test_static_cb is not None:
    test_static_cb_proc = process_dates(test_static_cb.join(test_base.select(["case_id", "date_decision"]),on="case_id", how="left"))
    test_static_cb_proc = test_static_cb_proc.drop("date_decision")
    del test_static_cb
else:
    test_static_cb_proc = None

test_tables_1n = {
    "person1":          test_person_1,
    "applprev_1":       test_applprev_1,
    # "applprev_2":       test_applprev_2,
    "debitcard":        test_debitcard,
    "deposit":          test_deposit,
    "tax_reg_a":        test_tax_registry_a_1,
    "tax_reg_b":        test_tax_registry_b_1,
    "bureau_a_1":       test_credit_bureau_a_1,
}

test_table, _ = feature_engineering(test_base, test_tables_1n, schema_dict=saved_schema)

# 3. Join an toàn
test_data = test_base
if test_static_proc is not None:
    test_data = test_data.join(test_static_proc, on="case_id", how="left")
if test_static_cb_proc is not None:
    test_data = test_data.join(test_static_cb_proc, on="case_id", how="left")
if test_table is not None:
    test_data = test_data.join(test_table, on="case_id", how="left")

if "date_decision" in test_data.columns:
    test_data = test_data.drop("date_decision")

print(f"Test shape: {test_data.shape}")
gc.collect()

Test shape: (10, 486)


0

In [18]:
test_data  = shrink_dtypes(test_data)

In [19]:
train_data, test_data = filter_features(train_data, test_data)
train_data, test_data = filter_correlated_features(train_data, test_data)
print(train_data.shape)
print(test_data.shape)

Null Filter: Dropping 121 columns with >95% nulls
Correlation Filter: Dropping 112 columns
(1526659, 254)
(10, 253)


In [20]:
common_cols = [c for c in train_data.columns if c in test_data.columns]
for col in common_cols:
    if train_data[col].dtype != test_data[col].dtype:
        print(f"{col} Train: {train_data[col].dtype}, Test:  {test_data[col].dtype}")

Saves the fully preprocessed training and test DataFrames to disk as Parquet files.

In [21]:
train_data.write_parquet(CURATED_DIR / "train_data.parquet")
test_data.write_parquet(CURATED_DIR / "test_data.parquet")

In [22]:
# train_data = pd.read_parquet("/kaggle/input/datasets/phucquangvo/data-curated/train_data.parquet")
# test_data = pd.read_parquet("/kaggle/input/datasets/phucquangvo/data-curated/test_data.parquet")

### Data Partitioning & Data Conversion
This section performs a robust train-validation split based on unique `case_id`s to prevent data leakage across different records of the same customer. It then extracts the predictive features, smoothly transitions the data structures from Polars to Pandas, and finalizes the categorical string encoding for model training.

In [23]:
case_ids = train_data["case_id"].unique().shuffle(seed=1)
case_ids_train, case_ids_val = train_test_split(case_ids, train_size=0.8, random_state=RANDOM_STATE)

cols_pred = []
for col in train_data.columns:
    if col not in meta_cols:
        cols_pred.append(col)

# print(cols_pred)

def from_polars_to_pandas(case_ids: pl.DataFrame) -> pl.DataFrame:
    ids = case_ids.to_list()
    df_filtered = train_data.filter(pl.col("case_id").is_in(ids))
    return (
        df_filtered[["case_id", "WEEK_NUM","MONTH","target"]].to_pandas(),
        df_filtered[cols_pred].to_pandas(),
        df_filtered["target"].to_pandas().squeeze()
    )

base_train, X_train, y_train = from_polars_to_pandas(case_ids_train)
base_valid, X_val, y_val = from_polars_to_pandas(case_ids_val)

X_train = convert_strings(X_train)
X_val = convert_strings(X_val)

Debug loop to catch any `category` datatype mismatches between `X_train` and `X_val`

In [24]:
cat_cols = X_train.select_dtypes(include=['category']).columns

for col in cat_cols:                
    if X_train[col].dtype.name != 'category' or X_val[col].dtype.name != 'category':
        print(f"{col}: X_train={X_train[col].dtype}, X_val={X_val[col].dtype}")


### LightGBM, XGBoost & CatBoost Training
This core section defines the K-Fold Stratified Cross-Validation pipelines for LightGBM, XGBoost, and CatBoost. Each function manages the fold splitting, handles model-specific dataset formatting, trains the model with early stopping, evaluates the Out-Of-Fold (OOF) AUC score, logs the metrics, and saves the trained model artifacts for future inference.

In [25]:
def lgb_cross_validate(X, y, categorical_features=None):
    def helper(X_train_fold, X_val_fold, y_train_fold, y_val_fold):
        train_data = lgb.Dataset(X_train_fold, label=y_train_fold, categorical_feature=categorical_features, free_raw_data=True)
        val_data   = lgb.Dataset(X_val_fold,   label=y_val_fold,   categorical_feature=categorical_features, free_raw_data=True, reference=train_data)
        
        model = lgb.train(
            LGBM_PARAMS,
            train_data,
            num_boost_round=NUM_BOOST_ROUND,
            valid_sets=[train_data, val_data],
            valid_names=['train', 'valid'],
            callbacks=[
                lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                lgb.log_evaluation(period=100)
            ]
        )
        del train_data, val_data
        gc.collect()

        return model

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    models = []
    oof_preds = np.zeros(len(X))
    cv_scores = []
    path = LOGS_DIR / f"lgbm_fold.txt"

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n{'-'*25} Fold {fold+1}/{N_FOLDS} {'-'*25}")

        # Filter to get train and validate dataset by its index
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # Evaluate
        model = helper(X_train, X_val, y_train, y_val)
        models.append(model)
        
        y_preds = model.predict(X_val, num_iteration=model.best_iteration)
        oof_preds[val_idx] = y_preds

        fold_auc = roc_auc_score(y_val, y_preds)
        cv_scores.append(fold_auc)
        print(f"Fold {fold+1} AUC: {fold_auc:.6f}")

        # Save 
        write_header = not path.exists()

        with open(path, 'a') as f:
            writer = csv.writer(f)
            if write_header:
                writer.writerow(['model_name', 'fold', 'auc', 'gini'])
            writer.writerow(['LightGBM', fold + 1, f"{fold_auc:.6f}", f"{2*fold_auc-1:.6f}"])
            
        model.save_model(str(MODELS_DIR / f"lgbm_fold_{fold}.txt"))
        gc.collect()

    oof_auc = roc_auc_score(y, oof_preds)
    with open(path, 'a') as f:
        writer = csv.writer(f)
        writer.writerow(['LightGBM', N_FOLDS + 1, f"{oof_auc:.6f}", f"{2*oof_auc-1:.6f}"])
    
    print(f"\nOOF AUC: {oof_auc:.6f}")
    print(f"Mean CV: {np.mean(cv_scores):6f} ± {np.std(cv_scores):6f}")

    return models, oof_preds

In [26]:
def xgb_cross_validate(X, y, categorical_features=None):
    def helper(X_train_fold, X_val_fold, y_train_fold, y_val_fold):
        train_data = xgb.DMatrix(X_train_fold, label=y_train_fold, enable_categorical=True)
        val_data   = xgb.DMatrix(X_val_fold,   label=y_val_fold,   enable_categorical=True)
        
        model = xgb.train(
            XGB_PARAMS,
            train_data,
            num_boost_round=NUM_BOOST_ROUND,
            evals=[(train_data, 'train'), (val_data, 'valid')],
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            verbose_eval=100
        )
        del train_data, val_data
        gc.collect()

        return model

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    models = []
    oof_preds = np.zeros(len(X))
    cv_scores = []
    path = LOGS_DIR / f"xgb_fold.txt"

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n{'-'*25} Fold {fold+1}/{N_FOLDS} {'-'*25}")

        # Filter to get train and validate dataset by its index
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # Evaluate
        model = helper(X_train, X_val, y_train, y_val)
        models.append(model)
        
        y_preds = model.predict(xgb.DMatrix(X_val, enable_categorical=True), iteration_range=(0, model.best_iteration))
        oof_preds[val_idx] = y_preds

        fold_auc = roc_auc_score(y_val, y_preds)
        cv_scores.append(fold_auc)
        print(f"Fold {fold+1} AUC: {fold_auc:.6f}")

        # Save 
        write_header = not path.exists()

        with open(path, 'a') as f:
            writer = csv.writer(f)
            if write_header:
                writer.writerow(['model_name', 'fold', 'auc', 'gini'])
            writer.writerow(['XGBoost', fold + 1, f"{fold_auc:.6f}", f"{2*fold_auc-1:.6f}"])
            
        model.save_model(str(MODELS_DIR / f"xgb_fold_{fold}.json"))
        gc.collect()

    oof_auc = roc_auc_score(y, oof_preds)
    with open(path, 'a') as f:
        writer = csv.writer(f)
        writer.writerow(['XGBoost', N_FOLDS + 1, f"{oof_auc:.6f}", f"{2*oof_auc-1:.6f}"])
    
    print(f"\nOOF AUC: {oof_auc:.6f}")
    print(f"Mean CV: {np.mean(cv_scores):6f} ± {np.std(cv_scores):6f}")

    return models, oof_preds

In [27]:
def cat_cross_validate(X, y, categorical_features=None):
    if categorical_features is None:
        categorical_features = X.select_dtypes(include=['category']).columns.tolist()
        
    def helper(X_train_fold, X_val_fold, y_train_fold, y_val_fold):
        for col in categorical_features:
            X_train_fold[col] = X_train_fold[col].fillna("Unknown")
            X_val_fold[col] = X_val_fold[col].fillna("Unknown")
        
        train_data = Pool(X_train_fold, label=y_train_fold, cat_features=categorical_features)
        val_data   = Pool(X_val_fold,   label=y_val_fold,   cat_features=categorical_features)
        
        model = CatBoostClassifier(
            **CATBOOST_PARAMS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            use_best_model=True,
        )
        model.fit(train_data, eval_set=val_data)

        del train_data, val_data
        gc.collect()

        return model

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    models = []
    oof_preds = np.zeros(len(X))
    cv_scores = []
    path = LOGS_DIR / f"cat_fold.txt"

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n{'-'*25} Fold {fold+1}/{N_FOLDS} {'-'*25}")

        # Filter to get train and validate dataset by its index
        X_train, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        # Evaluate
        model = helper(X_train, X_val, y_train, y_val)
        models.append(model)
        
        y_preds = model.predict(X_val, prediction_type='Probability')[:, 1]
        oof_preds[val_idx] = y_preds

        fold_auc = roc_auc_score(y_val, y_preds)
        cv_scores.append(fold_auc)
        print(f"Fold {fold+1} AUC: {fold_auc:.6f}")

        # Save 
        write_header = not path.exists()

        with open(path, 'a') as f:
            writer = csv.writer(f)
            if write_header:
                writer.writerow(['model_name', 'fold', 'auc', 'gini'])
            writer.writerow(['CatBoost', fold + 1, f"{fold_auc:.6f}", f"{2*fold_auc-1:.6f}"])
            
        model.save_model(str(MODELS_DIR / f"cat_fold_{fold}.cbm"))
        gc.collect()

    oof_auc = roc_auc_score(y, oof_preds)
    with open(path, 'a') as f:
        writer = csv.writer(f)
        writer.writerow(['CatBoost', N_FOLDS + 1, f"{oof_auc:.6f}", f"{2*oof_auc-1:.6f}"])
    
    print(f"\nOOF AUC: {oof_auc:.6f}")
    print(f"Mean CV: {np.mean(cv_scores):6f} ± {np.std(cv_scores):6f}")

    return models, oof_preds

### Training LightGBM

In [28]:
lgb_models, base_train["score_lgb"] = lgb_cross_validate(X_train, y_train)


------------------------- Fold 1/5 -------------------------
[100]	train's auc: 0.829211	valid's auc: 0.816758
[200]	train's auc: 0.845296	valid's auc: 0.824864
[300]	train's auc: 0.856073	valid's auc: 0.827728
[400]	train's auc: 0.86475	valid's auc: 0.828554
[500]	train's auc: 0.872537	valid's auc: 0.828561
[600]	train's auc: 0.879824	valid's auc: 0.82905
[700]	train's auc: 0.886447	valid's auc: 0.828938
Fold 1 AUC: 0.829105

------------------------- Fold 2/5 -------------------------
[100]	train's auc: 0.828935	valid's auc: 0.816072
[200]	train's auc: 0.844824	valid's auc: 0.824394
[300]	train's auc: 0.855755	valid's auc: 0.827435
[400]	train's auc: 0.864707	valid's auc: 0.827984
[500]	train's auc: 0.872557	valid's auc: 0.828433
[600]	train's auc: 0.879924	valid's auc: 0.828924
[700]	train's auc: 0.886473	valid's auc: 0.829044
[800]	train's auc: 0.892427	valid's auc: 0.828974
Fold 2 AUC: 0.829102

------------------------- Fold 3/5 -------------------------
[100]	train's auc: 0.828

### Training XGBoost

In [29]:
xgb_models, base_train["score_xgb"] = xgb_cross_validate(X_train, y_train)


------------------------- Fold 1/5 -------------------------
[0]	train-auc:0.71993	valid-auc:0.72167
[100]	train-auc:0.83647	valid-auc:0.81890
[200]	train-auc:0.85197	valid-auc:0.82548
[300]	train-auc:0.86143	valid-auc:0.82847
[400]	train-auc:0.86931	valid-auc:0.83033
[500]	train-auc:0.87585	valid-auc:0.83142
[600]	train-auc:0.88201	valid-auc:0.83202
[700]	train-auc:0.88718	valid-auc:0.83246
[800]	train-auc:0.89202	valid-auc:0.83279
[900]	train-auc:0.89662	valid-auc:0.83300
[976]	train-auc:0.89991	valid-auc:0.83301
Fold 1 AUC: 0.833055

------------------------- Fold 2/5 -------------------------
[0]	train-auc:0.71537	valid-auc:0.71049
[100]	train-auc:0.83564	valid-auc:0.81794
[200]	train-auc:0.85116	valid-auc:0.82599
[300]	train-auc:0.86140	valid-auc:0.82933
[400]	train-auc:0.86916	valid-auc:0.83097
[500]	train-auc:0.87578	valid-auc:0.83197
[600]	train-auc:0.88184	valid-auc:0.83265
[700]	train-auc:0.88720	valid-auc:0.83307
[800]	train-auc:0.89213	valid-auc:0.83336
[900]	train-auc:0.8

### Training CatBoost

In [30]:
cat_models, base_train["score_cat"] = cat_cross_validate(X_train, y_train)


------------------------- Fold 1/5 -------------------------


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7152897	best: 0.7152897 (0)	total: 378ms	remaining: 15m 44s
100:	test: 0.8081652	best: 0.8081652 (100)	total: 23.4s	remaining: 9m 15s
200:	test: 0.8173453	best: 0.8173453 (200)	total: 45.9s	remaining: 8m 44s
300:	test: 0.8218469	best: 0.8218469 (300)	total: 1m 8s	remaining: 8m 16s
400:	test: 0.8247922	best: 0.8247922 (400)	total: 1m 30s	remaining: 7m 51s
500:	test: 0.8265442	best: 0.8265442 (500)	total: 1m 51s	remaining: 7m 25s
600:	test: 0.8277887	best: 0.8277898 (599)	total: 2m 13s	remaining: 7m 2s
700:	test: 0.8286449	best: 0.8286449 (700)	total: 2m 35s	remaining: 6m 39s
800:	test: 0.8291712	best: 0.8291712 (800)	total: 2m 57s	remaining: 6m 16s
900:	test: 0.8296325	best: 0.8296325 (900)	total: 3m 19s	remaining: 5m 53s
1000:	test: 0.8300374	best: 0.8300725 (994)	total: 3m 41s	remaining: 5m 31s
1100:	test: 0.8303389	best: 0.8303605 (1096)	total: 4m 2s	remaining: 5m 8s
1200:	test: 0.8306383	best: 0.8306479 (1195)	total: 4m 24s	remaining: 4m 46s
1300:	test: 0.8309208	best: 0.

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7095157	best: 0.7095157 (0)	total: 261ms	remaining: 10m 51s
100:	test: 0.8064875	best: 0.8064875 (100)	total: 23.5s	remaining: 9m 17s
200:	test: 0.8176517	best: 0.8176517 (200)	total: 46.4s	remaining: 8m 50s
300:	test: 0.8226721	best: 0.8226721 (300)	total: 1m 8s	remaining: 8m 23s
400:	test: 0.8257794	best: 0.8257794 (400)	total: 1m 30s	remaining: 7m 55s
500:	test: 0.8277071	best: 0.8277071 (500)	total: 1m 52s	remaining: 7m 29s
600:	test: 0.8289046	best: 0.8289046 (600)	total: 2m 14s	remaining: 7m 5s
700:	test: 0.8297838	best: 0.8297889 (696)	total: 2m 36s	remaining: 6m 42s
800:	test: 0.8303700	best: 0.8303766 (799)	total: 2m 58s	remaining: 6m 19s
900:	test: 0.8308368	best: 0.8308368 (900)	total: 3m 20s	remaining: 5m 56s
1000:	test: 0.8311137	best: 0.8311137 (1000)	total: 3m 42s	remaining: 5m 33s
1100:	test: 0.8314318	best: 0.8314318 (1100)	total: 4m 4s	remaining: 5m 10s
1200:	test: 0.8318316	best: 0.8318480 (1196)	total: 4m 26s	remaining: 4m 48s
1300:	test: 0.8318812	best: 

...

0:	test: 0.7108496	best: 0.7108496 (0)	total: 269ms	remaining: 11m 11s
100:	test: 0.8059283	best: 0.8059283 (100)	total: 23.8s	remaining: 9m 24s
200:	test: 0.8167222	best: 0.8167222 (200)	total: 46.3s	remaining: 8m 49s
300:	test: 0.8217261	best: 0.8217261 (300)	total: 1m 8s	remaining: 8m 21s
400:	test: 0.8247709	best: 0.8247709 (400)	total: 1m 30s	remaining: 7m 54s
500:	test: 0.8266612	best: 0.8266612 (500)	total: 1m 52s	remaining: 7m 29s
600:	test: 0.8280466	best: 0.8280466 (600)	total: 2m 14s	remaining: 7m 6s
700:	test: 0.8291463	best: 0.8291537 (699)	total: 2m 37s	remaining: 6m 43s
800:	test: 0.8300566	best: 0.8300566 (800)	total: 2m 59s	remaining: 6m 20s
900:	test: 0.8306538	best: 0.8306746 (897)	total: 3m 21s	remaining: 5m 57s
1000:	test: 0.8312353	best: 0.8312353 (1000)	total: 3m 43s	remaining: 5m 34s
1100:	test: 0.8314186	best: 0.8314389 (1099)	total: 4m 5s	remaining: 5m 11s
1200:	test: 0.8317196	best: 0.8317326 (1195)	total: 4m 26s	remaining: 4m 48s
1300:	test: 0.8319339	best: 

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7117341	best: 0.7117341 (0)	total: 264ms	remaining: 11m
100:	test: 0.8043665	best: 0.8043665 (100)	total: 23.5s	remaining: 9m 18s
200:	test: 0.8154727	best: 0.8154727 (200)	total: 46.3s	remaining: 8m 49s
300:	test: 0.8203368	best: 0.8203368 (300)	total: 1m 8s	remaining: 8m 21s
400:	test: 0.8235361	best: 0.8235361 (400)	total: 1m 30s	remaining: 7m 55s
500:	test: 0.8252724	best: 0.8252724 (500)	total: 1m 52s	remaining: 7m 29s
600:	test: 0.8267506	best: 0.8267507 (599)	total: 2m 14s	remaining: 7m 5s
700:	test: 0.8277854	best: 0.8278318 (698)	total: 2m 36s	remaining: 6m 42s
800:	test: 0.8284526	best: 0.8284526 (800)	total: 2m 58s	remaining: 6m 18s
900:	test: 0.8290704	best: 0.8290704 (900)	total: 3m 20s	remaining: 5m 56s
1000:	test: 0.8293307	best: 0.8293588 (995)	total: 3m 42s	remaining: 5m 33s
1100:	test: 0.8296993	best: 0.8297119 (1091)	total: 4m 4s	remaining: 5m 10s
1200:	test: 0.8300059	best: 0.8300059 (1200)	total: 4m 26s	remaining: 4m 48s
1300:	test: 0.8302995	best: 0.830

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.7141087	best: 0.7141087 (0)	total: 264ms	remaining: 10m 58s
100:	test: 0.8079318	best: 0.8079318 (100)	total: 23.4s	remaining: 9m 16s
200:	test: 0.8183400	best: 0.8183400 (200)	total: 46.3s	remaining: 8m 50s
300:	test: 0.8232131	best: 0.8232131 (300)	total: 1m 8s	remaining: 8m 21s
400:	test: 0.8261014	best: 0.8261014 (400)	total: 1m 30s	remaining: 7m 54s
500:	test: 0.8283958	best: 0.8283958 (500)	total: 1m 52s	remaining: 7m 30s
600:	test: 0.8297960	best: 0.8297960 (600)	total: 2m 14s	remaining: 7m 5s
700:	test: 0.8309106	best: 0.8309106 (700)	total: 2m 36s	remaining: 6m 42s
800:	test: 0.8315802	best: 0.8315847 (799)	total: 2m 58s	remaining: 6m 19s
900:	test: 0.8321729	best: 0.8321795 (899)	total: 3m 20s	remaining: 5m 56s
1000:	test: 0.8323902	best: 0.8324029 (996)	total: 3m 42s	remaining: 5m 33s
1100:	test: 0.8326196	best: 0.8326196 (1100)	total: 4m 4s	remaining: 5m 10s
1200:	test: 0.8328484	best: 0.8328484 (1200)	total: 4m 26s	remaining: 4m 48s
1300:	test: 0.8332414	best: 0

### Evaluation
This section generates predictions by averaging the outputs from all cross-validation folds for each algorithm. It then evaluates the models using the competition's custom **Gini Stability** metric, which scores predictive power while penalizing performance degradation and high variance over time.

In [31]:
def predict_cv(models, X_test, model_name):
    X_ptr = X_test.copy()
    
    if model_name == 'cat':
        cat_features = X_ptr.select_dtypes(include=['category']).columns.tolist()
        for col in cat_features:
            X_ptr[col] = X_ptr[col].fillna("Unknown")
            
    preds = np.zeros(len(X_ptr))
    for model in tqdm(models, desc=f"Predicting {model_name.upper()}", file=sys.stdout):
        if model_name == "lgb":
            preds += model.predict(X_ptr, num_iteration=model.best_iteration)
        elif model_name == "xgb":
            preds += model.predict(xgb.DMatrix(X_ptr, enable_categorical=True), iteration_range=(0, model.best_iteration))
        elif model_name == 'cat':
            preds += model.predict(X_ptr, prediction_type='Probability')[:, 1]
    return preds / len(models)

In [32]:
def gini_stability(base, score_col="score_lgb", w_fallingrate=88.0, w_resstd=-0.5):
    gini_in_time = base.loc[:, ["WEEK_NUM", "target", score_col]]\
        .sort_values("WEEK_NUM")\
        .groupby("WEEK_NUM")[["target", score_col]]\
        .apply(lambda x: 2*roc_auc_score(x["target"], x[score_col])-1).tolist()
    
    x = np.arange(len(gini_in_time))
    y = gini_in_time
    a, b = np.polyfit(x, y, 1)
    y_hat = a*x + b
    residuals = y - y_hat
    res_std = np.std(residuals)
    avg_gini = np.mean(gini_in_time)
    return avg_gini + w_fallingrate * min(0, a) + w_resstd * res_std
    
base_valid["score_lgb_val"] = predict_cv(lgb_models, X_val, "lgb")
base_valid["score_xgb_val"] = predict_cv(xgb_models, X_val, "xgb")
base_valid["score_cat_val"] = predict_cv(cat_models, X_val, "cat")

stability_score_train_lgb = gini_stability(base_train, score_col="score_lgb")
stability_score_train_xgb = gini_stability(base_train, score_col="score_xgb")
stability_score_train_cat = gini_stability(base_train, score_col="score_cat")

stability_score_valid_lgb = gini_stability(base_valid, score_col="score_lgb_val")
stability_score_valid_xgb = gini_stability(base_valid, score_col="score_xgb_val")
stability_score_valid_cat = gini_stability(base_valid, score_col="score_cat_val")

print("LightGBM")
print(f'The stability score on the train set is: {stability_score_train_lgb}')
print(f'The stability score on the valid set is: {stability_score_valid_lgb}')
print("XGBoost")
print(f'The stability score on the train set is: {stability_score_train_xgb}')
print(f'The stability score on the valid set is: {stability_score_valid_xgb}')
print("CatBoost")
print(f'The stability score on the train set is: {stability_score_train_cat}')
print(f'The stability score on the valid set is: {stability_score_valid_cat}')

Predicting CAT: 100%|██████████| 5/5 [00:07<00:00,  1.56s/it]
LightGBM
The stability score on the train set is: 0.631501600774319
The stability score on the valid set is: 0.6118667765236947
XGBoost
The stability score on the train set is: 0.6402801046805531
The stability score on the valid set is: 0.6167718566781684
CatBoost
The stability score on the train set is: 0.637485575471087
The stability score on the valid set is: 0.6189050364283114


In [33]:
# MODELS_DIR = "/kaggle/input/datasets/phucquangvo/models/models/"

# cat_models = []
# lgb_models = []
# xgb_models = []

# for fold in range(N_FOLDS):
#     # 1. Load CatBoost (.cbm)
#     cat_model = CatBoostClassifier()
#     cat_model.load_model(str(MODELS_DIR + f"cat_fold_{fold}.cbm"))
#     cat_models.append(cat_model)
    
#     # 2. Load LightGBM (.txt)
#     lgb_model = lgb.Booster(model_file=str(MODELS_DIR + f"lgbm_fold_{fold}.txt"))
#     lgb_models.append(lgb_model)
    
#     # 3. Load XGBoost (.json)
#     xgb_model = xgb.Booster()
#     xgb_model.load_model(str(MODELS_DIR + f"xgb_fold_{fold}.json"))
#     xgb_models.append(xgb_model)

In [34]:
base_train.to_parquet(CURATED_DIR / "base_train.parquet")
base_valid.to_parquet(CURATED_DIR / "base_valid.parquet")

### Kaggle Submission (Optional)
This block strictly aligns the test set's schema and categorical features with your training data, safely handling any missing columns or unseen categories. The pipeline is now fully complete, and you are ready to generate the final predictions and submit this model to Kaggle for scoring!

In [35]:
X_submission = test_data[cols_pred].to_pandas()

cat_cols = X_train.select_dtypes(include=['category']).columns
for col in cat_cols:
    X_submission[col] = pd.Categorical(
        X_submission[col], 
        categories=X_train[col].cat.categories, 
        ordered=True
    )
    X_submission[col] = X_submission[col].fillna("Unknown")

In [36]:
for col in cat_cols:                
    if X_train[col].dtype.name != 'category' or X_submission[col].dtype.name != 'category':
        print(f"{col}: X_train={X_train[col].dtype}, X_submission={X_submission[col].dtype}")


In [37]:
for col in cols_pred:
    if col not in test_data.columns:
        test_data = test_data.with_columns(pl.lit(None).cast(train_data[col].dtype).alias(col))

X_submission = test_data[cols_pred].to_pandas()
X_submission = convert_strings(X_submission)
print(len(X_train.columns))
print(len(X_submission.columns))

categorical_cols = X_train.select_dtypes(include=['category']).columns
print(len(X_train[categorical_cols].columns))
print(len(X_submission[categorical_cols].columns))

for col in categorical_cols:
    train_categories = set(X_train[col].cat.categories)
    submission_categories = set(X_submission[col].cat.categories)
    
    new_categories = submission_categories - train_categories
    if (new_categories):
        print(new_categories)    
    X_submission[col] = pd.Categorical(X_submission[col], categories=train_categories)
    X_submission[col] = X_submission[col].fillna("Unknown")

250
250
49
49


In [38]:
print(len([c for c in X_train.columns      if X_train[c].dtype.name == 'category']))
print(len([c for c in X_submission.columns if X_submission[c].dtype.name == 'category']))
print(len([c for c in X_train.columns      if X_train[c].dtype.name == 'category' and X_submission[c].dtype.name != 'category']))

49
49
0


In [39]:
preds_lgb = predict_cv(lgb_models, X_submission, "lgb")
preds_xgb = predict_cv(xgb_models, X_submission, "xgb")
preds_cat = predict_cv(cat_models, X_submission, "cat")

Predicting CAT: 100%|██████████| 5/5 [00:00<00:00, 157.79it/s]


In [40]:
final_preds = (0.35 * preds_lgb) + (0.15 * preds_xgb) + (0.50 * preds_cat)

In [41]:
print(final_preds)

[0.1394203  0.28340226 0.06004616 0.14758739 0.62356461 0.0336964
 0.22555676 0.00510789 0.25703085 0.06186628]


In [42]:
submission = pd.DataFrame({
    "case_id": test_data["case_id"],
    "score": final_preds
})

In [43]:
print(submission)

   case_id     score
0    57543  0.139420
1    57549  0.283402
2    57551  0.060046
3    57552  0.147587
4    57569  0.623565
5    57630  0.033696
6    57631  0.225557
7    57632  0.005108
8    57633  0.257031
9    57634  0.061866


In [44]:
submission.to_csv("submission.csv", index=False)